# Data-Level Sampling Strategies

This notebook compares **oversampling and hybrid resampling** techniques applied
to the training set only. The test set remains untouched. All models use the same
Random Forest with `class_weight='balanced'` and a tuned threshold of **0.21**.

## 1. Setup

In [1]:
import os
import sys
import time

import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

assert os.path.exists(os.path.join('..', 'data', 'Breast_Cancer.csv')), \
    "Dataset not found. Place Breast_Cancer.csv in the data/ folder."

from src.preprocessing import load_and_clean, split_features_target, get_train_test_split
from src.feature_engineering import add_features
from src.sampling import apply_sampler, plot_sampling_comparison
from src.models import build_model, train_model
from src.evaluation import evaluate_model, save_results

DATA_PATH = os.path.join('..', 'data', 'Breast_Cancer.csv')
FIGURES_DIR = os.path.join('..', 'results', 'figures')
METRICS_DIR = os.path.join('..', 'results', 'metrics')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

THRESHOLD = 0.21
METHODS = ['none', 'smote', 'adasyn', 'smoteenn', 'smotetomek']

df = load_and_clean(DATA_PATH)
df = add_features(df)
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = get_train_test_split(X, y)

## 2. Sampling strategies overview

| Method | Description |
|--------|-------------|
| **none** | No resampling — raw training distribution |
| **SMOTE** | Synthetic Minority Over-sampling — interpolates between minority neighbours |
| **ADASYN** | Adaptive Synthetic — focuses on harder-to-learn minority examples |
| **SMOTEENN** | SMOTE + Edited Nearest Neighbours cleaning |
| **SMOTETomek** | SMOTE + Tomek links undersampling of borderline majority samples |

## 3. PCA scatter comparison

In [2]:
plot_sampling_comparison(
    X_train, y_train, METHODS,
    save_path=os.path.join(FIGURES_DIR, '02_sampling_scatter.png')
)
print('Saved: results/figures/02_sampling_scatter.png')

Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}


Resampled shape: (5452, 30)
Resampled class counts: {1: 2726, 0: 2726}
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}
Resampled shape: (5507, 30)
Resampled class counts: {0: 2781, 1: 2726}
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}


Resampled shape: (3822, 30)
Resampled class counts: {0: 2363, 1: 1459}
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}
Resampled shape: (5346, 30)
Resampled class counts: {1: 2673, 0: 2673}


Saved: results/figures/02_sampling_scatter.png


## 4. Experiment loop

For each sampling method: resample training data → train RF → evaluate on the
**same held-out test set** at threshold 0.21.

In [3]:
sampling_results = []

for method in METHODS:
    print('=' * 60)
    print(f'TRAINING: RF + {method.upper()}')
    print('=' * 60)
    try:
        X_res, y_res = apply_sampler(X_train, y_train, method=method)
        model = build_model('vanilla_rf', class_weight='balanced')
        start = time.time()
        model = train_model(model, X_res, y_res)
        print(f'Trained in {time.time() - start:.1f}s')

        metrics = evaluate_model(
            f'RF + {method.upper()}', model, X_test, y_test, threshold=THRESHOLD
        )
        sampling_results.append(metrics)
        print(pd.Series(metrics))

        from src.evaluation import apply_threshold
        from src import visualization
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = apply_threshold(y_prob, threshold=THRESHOLD)
        safe_name = method.lower()
        visualization.plot_confusion_matrix(
            y_test, y_pred, f'RF + {method.upper()}',
            save_path=os.path.join(FIGURES_DIR, f'02_confusion_rf_{safe_name}.png')
        )
    except Exception as e:
        print(f'ERROR with method {method}: {e}')

comparison_df = pd.DataFrame(sampling_results)
print('\nComparison (sorted by recall_dead):')
display_cols = ['model_name', 'recall_dead', 'precision_dead', 'f1_dead',
                'f2_dead', 'false_negatives', 'pr_auc', 'roc_auc']
print(comparison_df[display_cols].sort_values('recall_dead', ascending=False).to_string(index=False))

TRAINING: RF + NONE
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}
Resampled shape: (3219, 30)
Resampled class counts: {1: 2726, 0: 493}


Trained in 4.3s


model_name                          RF + NONE
threshold                                0.21
accuracy                               0.7292
recall_dead                            0.4309
precision_dead                         0.2637
f1_dead                                0.3272
f2_dead                                0.3824
roc_auc                                0.7034
pr_auc                                 0.9246
false_negatives                            70
recall_alive                            0.783
precision_alive                        0.8841
f1_alive                               0.8305
timestamp          2026-06-16T13:20:17.331039
dtype: object


TRAINING: RF + SMOTE
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}
Resampled shape: (5452, 30)
Resampled class counts: {1: 2726, 0: 2726}


Trained in 4.8s


...

model_name                      RF + SMOTEENN
threshold                                0.21
accuracy                               0.4944
recall_dead                            0.8862
precision_dead                         0.2171
f1_dead                                0.3488
f2_dead                                0.5483
roc_auc                                 0.729
pr_auc                                 0.9344
false_negatives                            14
recall_alive                           0.4238
precision_alive                        0.9538
f1_alive                               0.5868
timestamp          2026-06-16T13:20:39.131943
dtype: object


TRAINING: RF + SMOTETOMEK
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}
Resampled shape: (5346, 30)
Resampled class counts: {1: 2673, 0: 2673}


Trained in 4.7s


model_name                    RF + SMOTETOMEK
threshold                                0.21
accuracy                               0.6186
recall_dead                            0.6748
precision_dead                         0.2371
f1_dead                                 0.351
f2_dead                                0.4929
roc_auc                                 0.702
pr_auc                                 0.9256
false_negatives                            40
recall_alive                           0.6085
precision_alive                        0.9121
f1_alive                                 0.73
timestamp          2026-06-16T13:20:46.473678
dtype: object



Comparison (sorted by recall_dead):
     model_name  recall_dead  precision_dead  f1_dead  f2_dead  false_negatives  pr_auc  roc_auc
  RF + SMOTEENN       0.8862          0.2171   0.3488   0.5483               14  0.9344   0.7290
     RF + SMOTE       0.6911          0.2443   0.3609   0.5060               38  0.9283   0.7040
RF + SMOTETOMEK       0.6748          0.2371   0.3510   0.4929               40  0.9256   0.7020
    RF + ADASYN       0.6423          0.2232   0.3312   0.4669               44  0.9270   0.6995
      RF + NONE       0.4309          0.2637   0.3272   0.3824               70  0.9246   0.7034


## 5. Save results

In [4]:
save_path = os.path.join(METRICS_DIR, 'sampling_results.csv')
save_results(sampling_results, save_path)
comparison_df

Results saved to ..\results\metrics\sampling_results.csv (5 rows)


,model_name,threshold,accuracy,recall_dead,precision_dead,f1_dead,f2_dead,roc_auc,pr_auc,false_negatives,recall_alive,precision_alive,f1_alive,timestamp
0,RF + NONE,0.21,0.7292,0.4309,0.2637,0.3272,0.3824,0.7034,0.9246,70,0.7830,0.8841,0.8305,2026-06-16T13:20:17.331039
1,RF + SMOTE,0.21,0.6261,0.6911,0.2443,0.3609,0.5060,0.7040,0.9283,38,0.6144,0.9168,0.7357,2026-06-16T13:20:25.105668
2,RF + ADASYN,0.21,0.6037,0.6423,0.2232,0.3312,0.4669,0.6995,0.9270,44,0.5968,0.9024,0.7184,2026-06-16T13:20:32.238487
3,RF + SMOTEENN,0.21,0.4944,0.8862,0.2171,0.3488,0.5483,0.7290,0.9344,14,0.4238,0.9538,0.5868,2026-06-16T13:20:39.131943
4,RF + SMOTETOMEK,0.21,0.6186,0.6748,0.2371,0.3510,0.4929,0.7020,0.9256,40,0.6085,0.9121,0.7300,2026-06-16T13:20:46.473678
